# Section 0 — PP-OCRv5 Mobile Kaggle baseline

**Experiment name:** PP-OCRv5 Mobile laptop-screenshot baseline

**Purpose:** Determine whether PP-OCRv5 Mobile can extract useful text and geometry from one real laptop screenshot, and measure its inference latency.

**Expected input:** One PNG, JPG, JPEG, or WebP laptop screenshot attached as a Kaggle dataset.

**Expected output:** Recognized text, confidence scores, polygons, a visualization, latency measurements, and a project-neutral JSON result in /kaggle/working/ocr_result.json.

**Model configuration:** PP-OCRv5_mobile_det for detection and latin_PP-OCRv5_mobile_rec for recognition. The Latin model supports Vietnamese, English, and other Latin-script languages. PP-OCRv6 is deliberately not used.

Laptop screenshots differ from scanned documents: they are already upright, flat, and digitally generated, but may contain very small text, several UI regions, source-code symbols, and mixed languages. Document orientation classification, document unwarping, and text-line orientation classification are therefore disabled for this baseline.

This notebook is a baseline experiment, not a production OCR service or an implementation of Screcaap's OCRProvider.

## Section 1 — Environment inspection

Inspect the Kaggle runtime before installing anything. Package versions are read from installation metadata so an older preinstalled package is not imported into memory before the pinned versions are installed.

In [ ]:
import importlib.metadata
import os
import platform
import shutil
import subprocess
import sys


def installed_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


print(f"Python version: {sys.version.split()[0]}")
print(f"Operating system: {platform.platform()}")
print(f"Kaggle runtime: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'not detected')}")

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    smi_result = subprocess.run(
        [nvidia_smi],
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )
    SYSTEM_HAS_NVIDIA_GPU = smi_result.returncode == 0
    print(f"Kaggle GPU enabled: {SYSTEM_HAS_NVIDIA_GPU}")
    print("\n--- nvidia-smi ---")
    print(smi_result.stdout if smi_result.stdout else smi_result.stderr)
else:
    SYSTEM_HAS_NVIDIA_GPU = False
    print("Kaggle GPU enabled: False")
    print("nvidia-smi: not available; enable a Kaggle GPU accelerator for GPU inference.")

print(f"PaddlePaddle CPU package: {installed_version('paddlepaddle')}")
print(f"PaddlePaddle GPU package: {installed_version('paddlepaddle-gpu')}")
print(f"PaddleOCR package: {installed_version('paddleocr')}")

## Section 2 — Dependency installation

This isolated cell installs only the PaddlePaddle inference engine and the base PaddleOCR package. It does not install PaddleOCR's optional document parsing, information extraction, translation, or training dependencies.

The GPU wheel must be compatible with the NVIDIA driver in the Kaggle image. This baseline uses PaddlePaddle's official stable CUDA 11.8 wheel, which has broad Kaggle driver compatibility. If Kaggle changes its image or driver, check the [official PaddlePaddle installation matrix](https://www.paddleocr.ai/main/en/version3.x/paddlepaddle_installation.html) before changing the wheel index. The base PaddleOCR install follows the [official general OCR installation guidance](https://www.paddleocr.ai/main/en/version3.x/installation.html).

Enable **Internet** in the Kaggle notebook settings: package installation and the first model initialization require downloads.

In [ ]:
# Pinned stable versions make the experiment repeatable.
PADDLE_VERSION = "3.2.2"
PADDLEOCR_VERSION = "3.7.0"

cpu_paddle = installed_version("paddlepaddle")
gpu_paddle = installed_version("paddlepaddle-gpu")
if cpu_paddle != "not installed" and gpu_paddle != "not installed":
    raise RuntimeError(
        "Both paddlepaddle and paddlepaddle-gpu are installed. Start a fresh Kaggle "
        "session (or remove one build) before continuing; Paddle does not support both together."
    )

if SYSTEM_HAS_NVIDIA_GPU:
    paddle_distribution = "paddlepaddle-gpu"
    paddle_index = "https://www.paddlepaddle.org.cn/packages/stable/cu118/"
    conflicting_distribution = "paddlepaddle"
else:
    paddle_distribution = "paddlepaddle"
    paddle_index = "https://www.paddlepaddle.org.cn/packages/stable/cpu/"
    conflicting_distribution = "paddlepaddle-gpu"

if installed_version(conflicting_distribution) != "not installed":
    raise RuntimeError(
        f"{conflicting_distribution} is already installed, but this runtime needs "
        f"{paddle_distribution}. Start a fresh Kaggle session and run top-to-bottom."
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        f"{paddle_distribution}=={PADDLE_VERSION}",
        "--index-url",
        paddle_index,
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        f"paddleocr=={PADDLEOCR_VERSION}",
    ]
)

print(f"Installed {paddle_distribution}: {installed_version(paddle_distribution)}")
print(f"Installed PaddleOCR: {installed_version('paddleocr')}")

On a fresh Kaggle session, no restart should be necessary because Section 1 did not import Paddle. If Paddle or PaddleOCR was imported earlier in this session, restart the Kaggle kernel after installation and run the notebook again from the top. The next cell imports the installed packages, reports their versions, and checks CUDA visibility with readable errors.

In [ ]:
import json
import time
from pathlib import Path

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import paddle
    import paddleocr
    import pandas as pd
    from IPython.display import display
    from PIL import Image, ImageDraw
except Exception as exc:
    raise RuntimeError(
        "The OCR environment could not be imported. Restart the Kaggle kernel if the "
        "installation cell replaced a package that had already been imported, then run all cells again."
    ) from exc

PADDLE_HAS_CUDA = bool(paddle.is_compiled_with_cuda())
PADDLE_GPU_COUNT = int(paddle.device.cuda.device_count()) if PADDLE_HAS_CUDA else 0

print(f"PaddlePaddle version: {paddle.__version__}")
print(f"PaddleOCR version: {paddleocr.__version__}")
print(f"PaddlePaddle detects CUDA: {PADDLE_HAS_CUDA}")
print(f"Detected Paddle GPU device count: {PADDLE_GPU_COUNT}")

if SYSTEM_HAS_NVIDIA_GPU and not PADDLE_HAS_CUDA:
    raise RuntimeError(
        "Kaggle exposes an NVIDIA GPU, but the installed PaddlePaddle build cannot use CUDA. "
        "Check the official PaddlePaddle wheel/driver compatibility matrix and rerun in a fresh session."
    )

## Section 3 — Configuration

Edit this one cell to choose the device, screenshot, recognition model, or benchmark length. latin_PP-OCRv5_mobile_rec is selected explicitly because it supports both Vietnamese and English Latin-script text. An alternative is PaddleOCR's lang="vi", ocr_version="PP-OCRv5" selection, but an explicit model name makes this baseline unambiguous.

In [ ]:
DEVICE = "gpu"  # Use "cpu" for a CPU-only Kaggle session.
IMAGE_PATH = None  # Example: "/kaggle/input/my-screenshots/screenshot.png"
WARMUP_RUNS = 1
BENCHMARK_RUNS = 5

OCR_VERSION = "PP-OCRv5"
DETECTION_MODEL = "PP-OCRv5_mobile_det"
RECOGNITION_MODEL = "latin_PP-OCRv5_mobile_rec"

if WARMUP_RUNS < 0 or BENCHMARK_RUNS < 1:
    raise ValueError("WARMUP_RUNS must be >= 0 and BENCHMARK_RUNS must be >= 1.")

if DEVICE == "gpu":
    EFFECTIVE_DEVICE = "gpu:0"
else:
    EFFECTIVE_DEVICE = DEVICE

if EFFECTIVE_DEVICE.startswith("gpu"):
    if not SYSTEM_HAS_NVIDIA_GPU:
        raise RuntimeError(
            "DEVICE requests GPU inference, but Kaggle has no GPU enabled. "
            "Enable an accelerator in Notebook options or change DEVICE to 'cpu'."
        )
    if not PADDLE_HAS_CUDA or PADDLE_GPU_COUNT < 1:
        raise RuntimeError(
            "DEVICE requests GPU inference, but PaddlePaddle cannot see a CUDA device. "
            "Check the installation output and Kaggle accelerator settings."
        )
elif EFFECTIVE_DEVICE != "cpu":
    raise ValueError("This baseline expects DEVICE to be 'gpu', 'gpu:0', or 'cpu'.")

print(f"Detector: {DETECTION_MODEL}")
print(f"Recognizer: {RECOGNITION_MODEL}")
print(f"Device: {EFFECTIVE_DEVICE}")

## Section 4 — Select an input screenshot

Kaggle datasets are mounted read-only under /kaggle/input. This cell recursively finds likely images, prints every candidate, automatically selects it when there is exactly one, and otherwise asks for an explicit path in the configuration cell. The image is loaded at its original dimensions and is not resized or overwritten.

In [ ]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
SUPPORTED_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp"}

if not KAGGLE_INPUT_DIR.exists():
    raise FileNotFoundError(
        "/kaggle/input does not exist. Run this notebook on Kaggle and attach a screenshot dataset."
    )

image_candidates = sorted(
    path
    for path in KAGGLE_INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
)

print(f"Discovered {len(image_candidates)} image file(s):")
for candidate in image_candidates:
    print(f"  {candidate}")

if IMAGE_PATH is None:
    if len(image_candidates) == 1:
        image_path = image_candidates[0]
        print(f"\nAutomatically selected the only image: {image_path}")
    elif not image_candidates:
        raise FileNotFoundError(
            "No supported screenshot was found under /kaggle/input. Attach a Kaggle dataset "
            "containing a .png, .jpg, .jpeg, or .webp image."
        )
    else:
        raise RuntimeError(
            "More than one image was found. Set IMAGE_PATH in Section 3 to one of the paths listed above."
        )
else:
    image_path = Path(IMAGE_PATH)

if not image_path.exists() or not image_path.is_file():
    raise FileNotFoundError(f"The selected IMAGE_PATH does not exist: {image_path}")
if image_path.suffix.lower() not in SUPPORTED_IMAGE_SUFFIXES:
    raise ValueError(f"Unsupported image extension: {image_path.suffix}")

try:
    image_path.resolve().relative_to(KAGGLE_INPUT_DIR.resolve())
except ValueError as exc:
    raise ValueError("IMAGE_PATH must point to an image under /kaggle/input.") from exc

with Image.open(image_path) as loaded_image:
    image_mode = loaded_image.mode
    original_image = loaded_image.copy()

image_width, image_height = original_image.size
print(f"Image path: {image_path}")
print(f"Width: {image_width}px")
print(f"Height: {image_height}px")
print(f"Mode: {image_mode}")
display(original_image)

## Section 5 — Initialize PP-OCRv5

Model construction is timed separately from inference. The first initialization may download model files, so initialization_ms can include network and setup time; it is reported for transparency but is never treated as normal per-screenshot latency. The pipeline is initialized once and reused.

In [ ]:
from paddleocr import PaddleOCR

initialization_started = time.perf_counter()
try:
    ocr = PaddleOCR(
        ocr_version=OCR_VERSION,
        text_detection_model_name=DETECTION_MODEL,
        text_recognition_model_name=RECOGNITION_MODEL,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        device=EFFECTIVE_DEVICE,
        engine="paddle_static",
    )
except Exception as exc:
    raise RuntimeError(
        "PP-OCRv5 initialization failed. Confirm that Kaggle Internet is enabled for the first "
        "model download and that the selected PaddlePaddle build matches the runtime device."
    ) from exc

initialization_ms = (time.perf_counter() - initialization_started) * 1000.0
print(f"Detection model: {DETECTION_MODEL}")
print(f"Recognition model: {RECOGNITION_MODEL}")
print(f"Device: {EFFECTIVE_DEVICE}")
print(f"Initialization time (may include downloads): {initialization_ms:.1f} ms")

## Section 6 — Run OCR and normalize the result

PaddleOCR's original result objects remain available in raw_ocr_results. For analysis, this cell reads the documented public .json result property and creates a small, provider-neutral ocr_blocks list. This is only an experiment about a possible future data shape; it does not define or implement the production OCRProvider contract.

In [ ]:
def to_builtin(value):
    """Recursively convert NumPy and Path values to JSON-safe Python values."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    return value


def public_result_payload(result):
    """Return the documented PaddleOCR 3.x JSON payload without private attributes."""
    if not hasattr(result, "json"):
        raise TypeError("This PaddleOCR result does not expose the documented public .json property.")
    data = result.json
    if callable(data):
        data = data()
    if isinstance(data, str):
        data = json.loads(data)
    data = to_builtin(data)
    if not isinstance(data, dict):
        raise TypeError(f"Expected a dictionary from result.json, received {type(data).__name__}.")
    payload = data.get("res", data)
    if not isinstance(payload, dict):
        raise TypeError("The public result JSON does not contain a dictionary payload.")
    return payload


raw_ocr_results = list(ocr.predict(str(image_path)))
if not raw_ocr_results:
    raise RuntimeError("PaddleOCR returned no result object for the selected screenshot.")

ocr_payloads = [public_result_payload(result) for result in raw_ocr_results]
ocr_blocks = []

for payload_index, payload in enumerate(ocr_payloads):
    texts = list(payload.get("rec_texts") or [])
    scores = list(payload.get("rec_scores") or [])
    polygons = payload.get("rec_polys")
    if polygons is None:
        polygons = payload.get("dt_polys")
    polygons = list(polygons or [])

    if not (len(texts) == len(scores) == len(polygons)):
        raise RuntimeError(
            "PaddleOCR returned mismatched recognized texts, scores, and polygons "
            f"for result {payload_index}: {len(texts)}, {len(scores)}, {len(polygons)}."
        )

    for text_value, score_value, polygon_value in zip(texts, scores, polygons):
        polygon = [[float(point[0]), float(point[1])] for point in polygon_value]
        if len(polygon) != 4:
            raise RuntimeError(f"Expected a four-point OCR polygon, received {len(polygon)} points.")
        ocr_blocks.append(
            {
                "text": str(text_value),
                "confidence": float(score_value),
                "polygon": polygon,
            }
        )

print(f"PaddleOCR result objects: {len(raw_ocr_results)}")
print(f"Normalized OCR blocks: {len(ocr_blocks)}")
ocr_blocks[:3]

## Section 7 — Human-readable OCR result

The table keeps text, confidence, and geometry together. combined_text is reconstructed by sorting blocks approximately from top to bottom and then left to right. This simple order is useful for inspection but will not perfectly model every multi-column desktop layout.

In [ ]:
def polygon_bounds(polygon):
    xs = [point[0] for point in polygon]
    ys = [point[1] for point in polygon]
    return min(xs), min(ys), max(xs), max(ys)


table_rows = []
for index, block in enumerate(ocr_blocks):
    x_min, y_min, x_max, y_max = polygon_bounds(block["polygon"])
    table_rows.append(
        {
            "index": index,
            "text": block["text"],
            "confidence": block["confidence"],
            "x_min": x_min,
            "y_min": y_min,
            "x_max": x_max,
            "y_max": y_max,
        }
    )

ocr_table = pd.DataFrame(
    table_rows,
    columns=["index", "text", "confidence", "x_min", "y_min", "x_max", "y_max"],
)
display(ocr_table)

reading_order_indices = sorted(
    range(len(ocr_blocks)),
    key=lambda index: (
        polygon_bounds(ocr_blocks[index]["polygon"])[1],
        polygon_bounds(ocr_blocks[index]["polygon"])[0],
    ),
)
combined_text = "\n".join(
    ocr_blocks[index]["text"] for index in reading_order_indices if ocr_blocks[index]["text"].strip()
)

print("Approximate top-to-bottom reading order:\n")
print(combined_text if combined_text else "[No recognized text]")

## Section 8 — Visualization

Draw each detected polygon and its table index on a copy. original_image remains unchanged.

In [ ]:
visualized_image = original_image.convert("RGB").copy()
drawer = ImageDraw.Draw(visualized_image)

for index, block in enumerate(ocr_blocks):
    polygon_points = [tuple(round(coordinate) for coordinate in point) for point in block["polygon"]]
    drawer.line(polygon_points + [polygon_points[0]], fill=(255, 45, 45), width=3)

    x_min, y_min, _, _ = polygon_bounds(block["polygon"])
    label = str(index)
    label_x = max(0, round(x_min))
    label_y = max(0, round(y_min) - 12)
    label_box = drawer.textbbox((label_x, label_y), label)
    drawer.rectangle(label_box, fill=(255, 45, 45))
    drawer.text((label_x, label_y), label, fill=(255, 255, 255))

plt.figure(figsize=(18, 12))
plt.imshow(visualized_image)
plt.title("PP-OCRv5 Mobile detections (labels match the table index)")
plt.axis("off")
plt.show()

## Section 9 — Latency benchmark

Initialization and model downloads are already complete. The benchmark performs the configured warm-up runs and then measures full end-to-end OCR prediction. GPU work is asynchronous, so Paddle is synchronized immediately before and after each timed run.

In [ ]:
from statistics import mean, median


def synchronize_device():
    if EFFECTIVE_DEVICE.startswith("gpu"):
        paddle.device.cuda.synchronize()


print(f"Running {WARMUP_RUNS} warm-up run(s)...")
for _ in range(WARMUP_RUNS):
    synchronize_device()
    _ = ocr.predict(str(image_path))
    synchronize_device()

inference_latencies_ms = []
print(f"Running {BENCHMARK_RUNS} measured run(s)...")
for run_index in range(BENCHMARK_RUNS):
    synchronize_device()
    started = time.perf_counter()
    _ = ocr.predict(str(image_path))
    synchronize_device()
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    inference_latencies_ms.append(elapsed_ms)
    print(f"Run {run_index + 1}: {elapsed_ms:.1f} ms")

mean_inference_ms = mean(inference_latencies_ms)
p50_inference_ms = median(inference_latencies_ms)

print("\nLatency summary (inference only):")
print(f"Mean: {mean_inference_ms:.1f} ms")
print(f"Median / p50: {p50_inference_ms:.1f} ms")
print(f"Min: {min(inference_latencies_ms):.1f} ms")
print(f"Max: {max(inference_latencies_ms):.1f} ms")

## Section 10 — Structured output

The final object keeps the selected engine and models, image metadata, every text block with its polygon, reconstructed text, and timing metrics. It is deliberately project-neutral. A shortened preview is printed, while the complete JSON is written only to Kaggle's working directory.

In [ ]:
normalized_result = {
    "engine": "paddleocr",
    "detector": DETECTION_MODEL,
    "recognizer": RECOGNITION_MODEL,
    "device": EFFECTIVE_DEVICE,
    "image": {
        "path": str(image_path),
        "width": int(image_width),
        "height": int(image_height),
    },
    "blocks": ocr_blocks,
    "combined_text": combined_text,
    "metrics": {
        "initialization_ms": float(initialization_ms),
        "p50_inference_ms": float(p50_inference_ms),
        "mean_inference_ms": float(mean_inference_ms),
    },
}
normalized_result = to_builtin(normalized_result)

preview_text = normalized_result["combined_text"]
if len(preview_text) > 500:
    preview_text = preview_text[:500] + "..."
preview = {
    **{key: value for key, value in normalized_result.items() if key not in {"blocks", "combined_text"}},
    "block_count": len(normalized_result["blocks"]),
    "blocks_preview": normalized_result["blocks"][:3],
    "combined_text_preview": preview_text,
}
print(json.dumps(preview, ensure_ascii=False, indent=2))

output_path = Path("/kaggle/working/ocr_result.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as output_file:
    json.dump(normalized_result, output_file, ensure_ascii=False, indent=2)

print(f"\nSaved full result to: {output_path}")

## Section 11 — What to inspect manually

- Did the detector miss small UI text?
- Did recognition preserve Vietnamese diacritics?
- Did it correctly recognize source-code punctuation and symbols?
- Did it confuse browser or sidebar text with the main content?
- Do confidence scores correspond to visibly correct and incorrect results?
- Would bounding boxes help the future semantic-analysis LLM understand screen regions?
- Is the measured latency reasonable for one screenshot approximately every 30 seconds?

## Next experiments

- PP-OCRv5 Mobile vs Server
- English vs Vietnamese/multilingual recognition
- Small UI text
- Source-code and terminal screenshots
- Preprocessing and downscaling experiments
- CPU vs GPU latency